**Do not rerun this notebook at the current point.**

It runs on non-anonymized files: the original Cozie `sample.csv` files (GPS columns `ws_latitude`, `ws_longitude`, `ws_altitude`), the original Oura `heartrate_*.csv` files and the raw AirCO2ntrol logger files. The shared repo only contains the anonymized copies (`*_anon.csv`). The outputs stored below come from an earlier run on the original data.

In [ ]:
from tqdm import tqdm
from datetime import datetime, timezone
import pandas as pd
import os, warnings
from pandas.errors import ParserWarning
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
warnings.simplefilter('ignore', ParserWarning)
import plotly.io as pio


from PersonAI.anchor import data_dir

# Final Analysis

In [ ]:
# data path
DATA_PATH = data_dir()

# Load the CSV files
df_final = pd.read_csv(os.path.join(DATA_PATH,"Aircontrol_with_SurveyAnswers_HR.csv"))

df_final = df_final[['user_id', 'q_preference_temperature']].groupby(['user_id', 'q_preference_temperature']).agg('size').reset_index(name='count').pivot(index='user_id', columns='q_preference_temperature', values='count')
df_final = (df_final.div(df_final.sum(axis=1), axis=0) * 100).round(0).reset_index()
df_final.columns.name=None

In [ ]:
# plot a Stacked Bar Chart using matplotlib
df_final.sort_values(['Keine Veränderung', 'Kühler', 'Wärmer']).plot(
    x = 'user_id',
    kind = 'barh',
    stacked = True,
    title = 'Participants Temperature Preference',
    mark_right = True,
    figsize=(12,18))

In [ ]:
df_final = pd.read_csv(os.path.join(DATA_PATH,'Aircontrol_with_SurveyAnswers_HR.csv'))
#don't know which file this should be
df = pd.read_csv(os.path.join(DATA_PATH,'cozie_oura_output.csv'))

In [ ]:
# localize survey_time, sensor_time, time_cozie
df_final['survey_time'] = pd.to_datetime(df_final['survey_time'], format='ISO8601')
df_final['survey_time'] = df_final['survey_time'].dt.tz_convert('Europe/Vienna')

df_final['sensor_time'] = pd.to_datetime(df_final['sensor_time'], format='ISO8601')
df_final['sensor_time'] = df_final['sensor_time'].dt.tz_convert('Europe/Vienna')

df['time_cozie'] = pd.to_datetime(df['time_cozie'], format='ISO8601')
df['time_cozie'] = df['time_cozie'].dt.tz_convert('Europe/Vienna')


In [ ]:
# outer join df <-> df_final on survey_time
df_merged_by_survey_time = pd.merge(left=df, right=df_final, how='outer', left_on=['time_cozie', 'id_participant'], right_on=['survey_time', 'user_id'])

In [ ]:
# see how many timestamp matches per user
df_final.groupby('user_id').agg('size')

In [ ]:
# for the matched timestamps check value matches in: location, feeling, heartbeat, preference temp
df_merged_by_survey_time[df_merged_by_survey_time['survey_time'].isna()].groupby('id_participant').agg('size').reset_index(name='count').sort_values('count')

In [ ]:
# for the matched timestamps check value matches in: location, feeling, heartbeat, preference temp
df_merged_by_survey_time[df_merged_by_survey_time['survey_time'].notna() &
    (
        (df_merged_by_survey_time['q_activity_x'] != df_merged_by_survey_time['q_activity_y']) |
        (df_merged_by_survey_time['q_location'] != df_merged_by_survey_time['q_location_x']) |
        (df_merged_by_survey_time['q_feeling_x'] != df_merged_by_survey_time['q_feeling_y']) |
        (df_merged_by_survey_time['q_heartbeat_x'] != df_merged_by_survey_time['q_heartbeat_y']) |
        (df_merged_by_survey_time['q_preference_temperature_x'] != df_merged_by_survey_time['q_preference_temperature_y'])
    )
]

In [ ]:
# check if sensor_time and survey_time generally match: answer no but usually seconds diff
df_merged_by_survey_time[df_merged_by_survey_time['survey_time'].notna() & (df_merged_by_survey_time['sensor_time'] != df_merged_by_survey_time['survey_time'])]

# Overview of participants' temp preferences

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH,"cozie_oura_output.csv"))
df = df[['id_participant', 'q_preference_temperature']].groupby(['id_participant', 'q_preference_temperature']).agg('size').reset_index(name='count').pivot(index='id_participant', columns='q_preference_temperature', values='count')
df = (df.div(df.sum(axis=1), axis=0) * 100).round(0).reset_index()
df.columns.name=None

In [ ]:
# plot a Stacked Bar Chart using matplotlib
df.sort_values(['Keine Veränderung', 'Kühler', 'Wärmer']).plot(
    x = 'id_participant',
    kind = 'barh',
    stacked = True,
    title = 'Participants Temperature Preference',
    mark_right = True,
    figsize=(12,18))

# Original dataset

## Notes
* Exclude users 29, 35, 40, 68 because they did not take part in the complete study
* No ANKILO: user 51
* Exclude id014, id015, id018, id019 and id041 because they are test users

In [ ]:
exclude_user_ids = ['id014', 'id029', 'id035', 'id040', 'id068', 'id015', 'id018', 'id019', 'id041']

cozie_dtypes = {
    'index': 'object',
    'id_participant': 'str',
    'q_activity': 'str',
    'q_location': 'str',
    'q_feeling': 'str',
    'q_heartbeat': 'str',
    'q_preference_temperature': 'str',
    'ws_latitude': 'float',
    'ws_longitude': 'float',
    'ws_altitude': 'float'
}

oura_dtypes = {
    'timestamp': 'object',
    'bpm': 'int'
}

cozie_usecols = list(cozie_dtypes.keys())
oura_usecols = list(oura_dtypes.keys())

# Only survey responses during certain activitie are selected
q_activity_vals_to_keep = [
    'Erholungsphase (Lesen, Fernsehen, etc.)',
    'Schreibtischarbeit (Bürotätigkeiten, etc.)',
    'Sonstige Aktivität'
]
# Only survey responses at certain locations are selected
q_location_vals_to_keep = [
    'Arbeitsplatz',
    'Zuhause'
]

## Read and pre-process data

In [ ]:
df = pd.DataFrame({col: pd.Series(dtype=dt) for col, dt in {**cozie_dtypes, **oura_dtypes}.items()})

users_without_sample = []

# Data are imported for each user separately and appended to full dataframe
for user in tqdm(os.listdir('Data')):
    if not user.startswith('id0') or user in exclude_user_ids:
        continue
    
    cozie_file_path = os.path.join('Data', user, 'Cozie', 'sample.csv')
    oura_file_path = next((os.path.join('Data', user, 'Oura', f) for f in os.listdir(os.path.join('Data', user, 'Oura')) if f.startswith('heartrate')), None)

    if not os.path.exists(cozie_file_path):
        users_without_sample.append(user)
        continue

    with open(cozie_file_path, 'r') as f:
        cozie_file_columns = f.readline().strip().split(',')

    # Some columns in cozie sample.csv are missing for some users
    cozie_avail_cols = [col for col in cozie_usecols if col in cozie_file_columns]
    cozie_dtype_subset = {col: cozie_dtypes[col] for col in cozie_avail_cols}
    cozie_missing_cols = [col for col in cozie_usecols if col not in cozie_avail_cols]

    df_cozie_data = pd.read_csv(
        cozie_file_path,
        dtype=cozie_dtype_subset,
        usecols=cozie_avail_cols,
        parse_dates=True,
        date_format='%Y-%m-%d %H:%M:%S.%f%z',
        sep=','
    )

    if oura_file_path:
        df_oura_hr_data = pd.read_csv(
            oura_file_path,
            dtype=oura_dtypes,
            usecols=oura_usecols,
            parse_dates=True,
            date_format='%Y-%m-%dT%H:%M:%S%z',
            sep=','
        )

    # Filter out the irrelevant activities and locations from Cozie
    mask = df_cozie_data['q_activity'].isin(q_activity_vals_to_keep) & df_cozie_data['q_location'].isin(q_location_vals_to_keep)
    indices = df_cozie_data[mask].index
    df_cozie_data = df_cozie_data.loc[indices]
    df_cozie_data.reset_index(inplace=True, drop=True)

    df_cozie_data['index'] = pd.to_datetime(df_cozie_data['index'])
    df_cozie_data['index'] = df_cozie_data['index'].dt.tz_convert('Europe/Vienna')
    df_cozie_data.sort_values(by=['index'], inplace=True)
    df_cozie_data.reset_index(inplace=True, drop=True)
    
    # Some users have 0 datapoints measured in oura ring
    if oura_file_path and len(df_oura_hr_data) > 0:
        # localize timestamps (originally in UTC)
        df_oura_hr_data['timestamp'] = pd.to_datetime(df_oura_hr_data['timestamp'])
        df_oura_hr_data['timestamp'] = df_oura_hr_data['timestamp'].dt.tz_convert('Europe/Vienna')
        df_cozie_data = pd.merge_asof(df_cozie_data, df_oura_hr_data, left_on='index', right_on='timestamp', direction='backward')
        
    else:
        for col in oura_usecols:
            df_cozie_data[col] = pd.NA
    
    for col in cozie_missing_cols:
        df_cozie_data[col] = pd.NA

    df = pd.concat([df, df_cozie_data], ignore_index=True)

df.rename(columns={'index':'time_cozie', 'timestamp': 'time_oura'}, inplace=True)
# df['time'] = pd.to_datetime(df['time'])
# df['time'] = df['time'].dt.tz_convert('Europe/Vienna')
df['id_participant'] = df['id_participant'].str.lower()
print(df.shape)
df.to_csv('cozie_oura_output.csv', encoding='UTF-8', index=False)

In [ ]:
df_counts = df.groupby("id_participant").size()
sample_size = df_counts.min()
print(f"Average number of responses per participant: {df_counts.mean()}")

## Import all sensor data matching Cozie Survey timestamps

In [ ]:
#don't know where find is
df_sensor_data = pd.read_csv(os.path.join(DATA_PATH,'merged_exact_time_and_id_clean.csv'))

In [ ]:
df_sensor_data.sort_values('survey_time',ascending=False)

## Downsample data to min (46) per participant

In [ ]:
def sample_equidistant(group, n_samples=60):
    if len(group) == n_samples:
        return pd.DataFrame()  # skip this group, or optionally handle differently
    # Select 46 equidistant indices
    indices = np.linspace(0, len(group) - 1, n_samples).astype(int)
    return group.iloc[indices]

# Apply to all users in the DataFrame
# df_sampled = df.groupby('id_participant', group_keys=False).apply(sample_equidistant).reset_index(drop=True)

## Data exploration
* Barchart, num of RHRN survey responses per participant, sorted, form downsampled dataset
* Linechart how many responses per participant per day across time, from downsampled dataset
* Box whisker plot, each feature is a box defined by its values, from downsampled dataset 
* Line chart: plot each feature on a separate chart, participants marked with unique colors, x is dates and y is feature value
* Scatterplot: air temp from aircontrol for all participants across time

### Number of RHRN Survey responses per participant from original dataset

In [ ]:
df_date_participant_response_count = df[['time_cozie', 'id_participant']].copy(deep=True)
df_date_participant_response_count['time_cozie'] = df_date_participant_response_count['time_cozie'].dt.date
df_date_participant_response_count = df_date_participant_response_count.groupby(['id_participant', 'time_cozie']).agg('size').reset_index(name='Count')
df_date_participant_response_count.rename(columns={
    'id_participant': 'Participant',
    'time_cozie': 'Date'
}, inplace=True)


In [ ]:
fig = px.scatter(
    df_date_participant_response_count,
    x='Date',
    y='Count',
    color='Participant',
    hover_name='Participant',
    title='Number of RHRN Survey responses per participant from original dataset'
)
fig.show()

### Number of RHRN Survey responses per participant from downsampled dataset

In [ ]:
df_date_participant_response_count = df_sampled[['time_cozie', 'id_participant']].copy(deep=True)
df_date_participant_response_count['time_cozie'] = df_date_participant_response_count['time_cozie'].dt.date
df_date_participant_response_count = df_date_participant_response_count.groupby(['id_participant', 'time_cozie']).agg('size').reset_index(name='Count')
df_date_participant_response_count.rename(columns={
    'id_participant': 'Participant',
    'time_cozie': 'Date'
}, inplace=True)

In [ ]:
fig = px.scatter(
    df_date_participant_response_count,
    x='Date',
    y='Count',
    color='Participant',
    hover_name='Participant',
    title='Number of RHRN Survey responses per participant from downsampled dataset'
)
fig.show()

In [ ]:
df_num_responses_per_participant = df[['id_participant']].copy(deep=True)
df_num_responses_per_participant = df_num_responses_per_participant.groupby('id_participant').agg('size').reset_index(name='Count')
df_num_responses_per_participant.rename(columns={'id_participant': 'Participant'}, inplace=True)

In [ ]:
fig = px.scatter(
    df_num_responses_per_participant,
    x='Participant',
    y='Count',
    color='Participant',
    hover_name='Participant',
    title='Number of survey responses per participant from original dataset'
)

avg_num_responses = df_num_responses_per_participant['Count'].mean()

fig.add_hline(
    y=avg_num_responses,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Average {avg_num_responses:.2f}',
    annotation_position='top left'
)
fig.show()

In [ ]:
df_bpm = df_sampled[['time_cozie', 'id_participant', 'bpm']].copy(deep=True)
df_bpm.rename(columns={
    'time_cozie': 'Time',
    'id_participant': 'Participant',
    'bpm': 'Heartrate(Oura)'
}, inplace=True)
fig = px.scatter(df_bpm, x='Time', y='Heartrate(Oura)', color='Participant', hover_name='Participant', opacity=.2)

avg_bpm = df_sampled['bpm'].mean()
q1 = df_sampled['bpm'].quantile(.25)
q3 = df_sampled['bpm'].quantile(.75)

fig.add_hline(
    y=avg_bpm,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Average {avg_bpm:.2f}',
    annotation_position='top left'
)

fig.add_hline(
    y=q1,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Q1 {q1:.2f}',
    annotation_position='top left'
)

fig.add_hline(
    y=q3,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Q3 {q3:.2f}',
    annotation_position='top left'
)

fig.show()

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH,"downsampled_sensors.csv"))

In [ ]:
df_full['survey_time'] = pd.to_datetime(df_full['survey_time'], format='%S:%M.%H')

In [ ]:
df_full.sort_values(by='survey_time', inplace=True)

In [ ]:
counts = result.groupby('user_id').agg('size').reset_index(name='Count')

In [ ]:
filtered_users = counts[counts['Count'] >= 60]['user_id']

In [ ]:
result_downsampled = result[result['user_id'].isin(filtered_users)].groupby('user_id', group_keys=False).apply(sample_equidistant).reset_index(drop=True)

In [ ]:
result_downsampled.to_csv(os.path.join(DATA_PATH,"final.csv"), encoding='UTF-8', index=False)

In [ ]:
result[result['user_id'].isin(filtered_users)]

In [ ]:
fig = px.scatter(result, x='time_cozie', y='AirTemp', color='user_id', hover_name='user_id', opacity=.4, title='Air Temperature (Celsius)')
avg_line = result['AirTemp'].mean()
fig.add_hline(
    y=avg_line,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg Air Temp {avg_line:.2f}',
    annotation_position='top left'
)

fig.show()

In [ ]:
fig = px.scatter(result, x='time_cozie', y='Humidity', color='user_id', hover_name='user_id', opacity=.4, title='Humidity (%)')
avg_line_humidity = result['Humidity'].mean()
fig.add_hline(
    y=avg_line_humidity,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg Humidty {avg_line_humidity:.2f}',
    annotation_position='top left'
)

fig.show()

In [ ]:
fig = px.scatter(result, x='time_cozie', y='CO2', color='user_id', hover_name='user_id', opacity=.4, title='CO2 ppm')
avg_line_co2 = result['CO2'].mean()
fig.add_hline(
    y=avg_line_co2,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg CO2 {avg_line_co2:.2f}',
    annotation_position='top left'
)

fig.show()

In [ ]:
fig = px.scatter(result, x='time_cozie', y='bpm', color='user_id', hover_name='user_id', opacity=.24, title='Heart Rate (Oura Ring)')
avg_line_bpm = result['bpm'].mean()
fig.add_hline(
    y=avg_line_bpm,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg Heartrate {avg_line_bpm:.2f}',
    annotation_position='top left'
)
avg_bpm = result['bpm'].mean()
q1 = result['bpm'].quantile(.25)
q3 = result['bpm'].quantile(.75)

fig.add_hline(
    y=q1,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Q1 {q1:.2f}',
    annotation_position='top left'
)

fig.add_hline(
    y=q3,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Q3 {q3:.2f}',
    annotation_position='top left'
)


fig.show()

In [ ]:
df_new = pd.read_csv(os.path.join(DATA_PATH,"Final_Aircontrol_Data.csv"), parse_dates=True, date_format='%Y-%m-%d %H:%M:%S.%f%z')

In [ ]:
df_new['survey_time'] = pd.to_datetime(df_new['survey_time'], format='ISO8601')

In [ ]:
df_new['sensor_time'] = pd.to_datetime(df_new['sensor_time'], format='ISO8601')

In [ ]:
df_new['time_difference'] = df_new['survey_time'] - df_new['sensor_time']

In [ ]:
df_new['time_difference'].describe()

In [ ]:
df = df[['time_cozie', 'time_oura', 'id_participant', 'q_activity',  'q_location', 'q_feeling', 'q_heartbeat', 'q_preference_temperature', 'bpm']]

In [ ]:
result = pd.merge(df_new, df, left_on='survey_time', right_on='time_cozie', how='left')

In [ ]:
result = result[['user_id', 'sensor_time', 'q_location_x', 'AirTemp', 'Humidity', 'CO2', 'time_cozie', 'time_oura', 'q_activity', 'q_location_y', 'q_feeling', 'q_heartbeat', 'q_preference_temperature', 'bpm']]

In [ ]:
result['time_difference_cozie_oura'] = result['time_cozie'] - result['time_oura']

In [ ]:
len(result['time_difference_cozie_oura'].dropna() <= pd.Timedelta(minutes=3)) / len(result['time_difference_cozie_oura'].dropna())


In [ ]:
(result['time_difference_cozie_oura'].dropna() <= pd.Timedelta(minutes=6)).sum()

In [ ]:
4501 / len(result['time_difference_cozie_oura'].dropna())

In [ ]:
result

# Test Users: compare AirControl and Room Temp/Humidity per user

Fetch AirControl temp/humidity values with utc time for users: id019, id014, id041, id018, id015 and filter only to Arbeitsplatz
Download and load room sensor temp/humidity values with utc time from LINK

Per User:
    Line Plot line1= Aircontrol temp, line2 Room sensor temp (if user= id014, line3= room sensor2 temp)
    Line plot line1= Aircontrol humidity, line2=room sensor humidity (if user= id014, line3= room sensor2 humidity)
*** Expected: constant offsent between aircontrol and room sensor data



In [ ]:
columns_humidity = {
    'user_id': 'str', 
    'relative humidity (%)': 'float', 
    'device': 'str'
}

columns_temperature = {
    'user_id': 'str', 
    'temperature (celsius)': 'float', 
    'device': 'str'
}
# cols in converters override cols above, so no need to put in both
converters = {
    'datetime_utc': lambda x: datetime.fromisoformat(x)
}

In [ ]:
def fetch_aircontrol_data_for_user(user):
    aircontrol_dir_name = f'Data/{user}/AirCO2ntrol'
    
    # only folders with even number in name must be processed
    aircontrol_dir_num = [num for num in os.listdir(aircontrol_dir_name) if int(num[1:]) % 2 == 0][0] 
    aircontrol_dir_name += f'/{aircontrol_dir_num}'

    # aircontrol data for a user is split across many csv files
    aircontrol_files = os.listdir(aircontrol_dir_name)
    
    aircontrol_df = pd.DataFrame()
    
    for file in aircontrol_files:
        df = pd.read_csv(
            f'{aircontrol_dir_name}/{file}',
            # usecols = ['D_M_YYYY','TIME[HH:mm:ss]','Temp[C]','RH[%]'],
            converters = {
                'D_M_YYYY': lambda x: datetime.strptime(x, '%d/%m/%Y'),
                'M_D_YYYY': lambda x: datetime.strptime(x, '%m/%d/%Y'),
                'TIME[HH:mm:ss]': lambda x: datetime.strptime(x, '%H:%M:%S').time()
            },
            sep=','
        )
        
        # Sometimes D_M_YYYY is (mistakenly) titled M_D_YYYY in the csv 
        if 'M_D_YYYY' in df.columns:
            df.rename(columns={'M_D_YYYY': 'D_M_YYYY'}, inplace=True)
        
        aircontrol_df = pd.concat([aircontrol_df, df], ignore_index=True)

    # aircontrol column selection and names are adjusted according to <columns_humidity> and <columns_temperature> (see block above)
    aircontrol_df['datetime_utc'] = aircontrol_df.apply(lambda row: datetime.combine(row['D_M_YYYY'].date(), row['TIME[HH:mm:ss]'], tzinfo=timezone.utc), axis=1)
    aircontrol_df.rename(columns={
        'Temp[C]': 'temperature (celsius)',
        'RH[%]': 'relative humidity (%)',
        'CO2[ppm]': 'co2 (ppm)'}, inplace=True)
    aircontrol_df['user_id'] = user
    aircontrol_df['device'] = 'air control'

    return aircontrol_df

def fetch_sensor_data_for_user(user, file_suffix, value_column_name):
    sensor_dfs = [pd.read_csv(
            f'room_sensors_forschung_bgld/{file}',
            dtype={
                'time': 'object',
                'value': 'float'
            },
            usecols=['time', 'value'],
            converters= {'time': lambda x: datetime.fromisoformat(x)},
            sep=','
        ) for file in os.listdir('room_sensors_forschung_bgld') if file.startswith(user) and file.endswith(f'_{file_suffix}.csv')]
    
    sensor_df = pd.concat(sensor_dfs, ignore_index=True)
    
    sensor_df.rename(columns={'time': 'datetime_utc', 'value': value_column_name}, inplace=True)
    sensor_df['datetime_utc'] = pd.to_datetime(sensor_df['datetime_utc'], utc=True)
    sensor_df['device'] = 'room sensor'
    sensor_df['user_id'] = user

    return sensor_df
    

In [ ]:
if 'humidity_aircontrol_room_sensor_comparison.csv' in os.listdir() and 'temperature_aircontrol_room_sensor_comparison.csv' in os.listdir():
    df_humidity = pd.read_csv(
        'humidity_aircontrol_room_sensor_comparison.csv',
        dtype=columns_humidity,
        converters=converters,
        sep=','        
    )

    df_temperature = pd.read_csv(
        'temperature_aircontrol_room_sensor_comparison.csv',
        dtype=columns_temperature,
        converters=converters,
        sep=','        
    )

else:
    
    df_humidity = pd.DataFrame({
        col: pd.Series(dtype=dtype) for col, dtype in columns_humidity.items()
    })
    
    df_temperature = pd.DataFrame({
        col: pd.Series(dtype=dtype) for col, dtype in columns_temperature.items()
    })
    
    users = ['id019', 'id014_1', 'id014_2', 'id041', 'id018', 'id015']
    
    for user in users:

        if user == 'id014_2' or user == 'id014_1':
            aircontrol_df = fetch_aircontrol_data_for_user(user[:-2])
            aircontrol_df['user_id'] = user
        else:
            aircontrol_df = fetch_aircontrol_data_for_user(user)

        df_sensor_humidity = fetch_sensor_data_for_user(user=user, value_column_name='relative humidity (%)', file_suffix='hum')
        df_sensor_temperature = fetch_sensor_data_for_user(user=user, value_column_name='temperature (celsius)', file_suffix='temp')
            
        df_humidity = pd.concat([df_humidity, aircontrol_df[['datetime_utc', 'relative humidity (%)', 'user_id', 'device']]], ignore_index=True)
        df_temperature = pd.concat([df_temperature, aircontrol_df[['datetime_utc', 'temperature (celsius)', 'user_id', 'device']]], ignore_index=True)
        df_humidity = pd.concat([df_humidity, df_sensor_humidity[['datetime_utc', 'relative humidity (%)', 'device', 'user_id']]])
        df_temperature = pd.concat([df_temperature, df_sensor_temperature[['datetime_utc', 'temperature (celsius)', 'device', 'user_id']]])
        
        # Exclude datapoints not in current year (there are quite a few due to different reasons)
        df_humidity = df_humidity[df_humidity['datetime_utc'] > pd.Timestamp('2025-01-01 00:00:01', tz='UTC')]
        df_temperature = df_temperature[df_temperature['datetime_utc'] > pd.Timestamp('2025-01-01 00:00:01', tz='UTC')]
    
    df_humidity.to_csv('humidity_aircontrol_room_sensor_comparison.csv', encoding='UTF-8', index=False) 
    df_temperature.to_csv('temperature_aircontrol_room_sensor_comparison.csv', encoding='UTF-8', index=False) 

In [ ]:
users = ['id019', 'id014_1', 'id014_2', 'id041', 'id018', 'id015']
offsets = {'hum': {}, 'temp': {}}
for user in users:
    
    temp_min_date = df_temperature[
        (df_temperature['user_id'] == user) & 
        (df_temperature['device'] == 'air control')
    ]['datetime_utc'].min()

    temp_max_date = df_temperature[
        (df_temperature['user_id'] == user) & 
        (df_temperature['device'] == 'air control')
    ]['datetime_utc'].max()

    hum_max_date = df_humidity[
        (df_humidity['user_id'] == user) & 
        (df_humidity['device'] == 'air control')
    ]['datetime_utc'].max()
    
    hum_min_date = df_humidity[
        (df_humidity['user_id'] == user) & 
        (df_humidity['device'] == 'air control')
    ]['datetime_utc'].min()

    df_humidity = df_humidity[(df_humidity['datetime_utc'] > hum_min_date) & (df_humidity['datetime_utc'] < hum_max_date)]
    df_temperature = df_temperature[(df_temperature['datetime_utc'] > temp_min_date) & (df_temperature['datetime_utc'] < temp_max_date)]
    
    temp_avg_sensor = float(df_temperature[(df_temperature['user_id']==user) & (df_temperature['device']=='room sensor')]['temperature (celsius)'].mean())
    temp_avg_aircontrol = float(df_temperature[(df_temperature['user_id']==user) & (df_temperature['device']=='air control')]['temperature (celsius)'].mean())
    hum_avg_sensor = float(df_humidity[(df_humidity['user_id']==user) & (df_humidity['device']=='room sensor')]['relative humidity (%)'].mean())
    hum_avg_aircontrol = float(df_humidity[(df_humidity['user_id']==user) & (df_humidity['device']=='air control')]['relative humidity (%)'].mean())

    offsets['temp'][user] = temp_avg_sensor - temp_avg_aircontrol
    offsets['hum'][user] = hum_avg_sensor - hum_avg_aircontrol

    fig = px.scatter(
        df_temperature[df_temperature['user_id']== user], 
        x='datetime_utc', 
        y='temperature (celsius)', 
        color='device', 
        opacity=0.2, 
        title= f'Comparison of room sensor and aircontrol temperature measurements for user {user}'
    )
    
    fig.add_hline(
        y=temp_avg_sensor,
        line_dash='dash',
        line_color='red',
        annotation_text=f'Avg sensor {temp_avg_sensor:.2f}',
        annotation_position='top left'
    )
    
    fig.add_hline(
        y=temp_avg_aircontrol,
        line_dash='dash',
        line_color='blue',
        annotation_text=f'Avg aircontrol {temp_avg_aircontrol:.2f}',
        annotation_position='top left'
    )
    
    fig.write_html(f'{user}_temperature.html')
    
    fig = px.scatter(
        df_humidity[df_humidity['user_id']==user], 
        x='datetime_utc', 
        y='relative humidity (%)', 
        color='device', 
        opacity=0.2, 
        title= f'Comparison of room sensor and aircontrol humidity measurements for user {user}'
    )
    
    fig.add_hline(
        y=hum_avg_sensor,
        line_dash='dash',
        line_color='red',
        annotation_text=f'Avg sensor {hum_avg_sensor:.2f}',
        annotation_position='top left'
    )
    
    fig.add_hline(
        y=hum_avg_aircontrol,
        line_dash='dash',
        line_color='blue',
        annotation_text=f'Avg aircontrol {hum_avg_aircontrol:.2f}',
        annotation_position='top left'
    )
    
    fig.write_html(f'{user}_humidity.html')

In [ ]:
print(f'Sensor humidity was on avg {sum(list(offsets['hum'].values())) / len(list(offsets['hum'].values())):.2f} percent points greater than aircontrol humidity')
print(f'Sensor temperature was on avg {sum(list(offsets['temp'].values())) / len(list(offsets['temp'].values())):.2f} celsius greater than aircontrol temperature')

In [ ]:
aircontrol_df = fetch_aircontrol_data_for_user('id019')

df_sensor_co2 = fetch_sensor_data_for_user(user='id019', value_column_name='co2 (ppm)', file_suffix='co2')
    
df_co2 = pd.concat([df_sensor_co2, aircontrol_df[['datetime_utc', 'co2 (ppm)', 'user_id', 'device']]], ignore_index=True)

# Exclude datapoints not in current year (there are quite a few due to different reasons)
df_co2 = df_co2[df_co2['datetime_utc'] > pd.Timestamp('2025-01-01 00:00:01', tz='UTC')]

df_co2.to_csv('co2_aircontrol_room_sensor_comparison.csv', encoding='UTF-8', index=False) 

In [ ]:
offsets = {'co2': {}}

co2_min_date = df_co2[
    (df_co2['user_id'] == 'id019') & 
    (df_co2['device'] == 'air control')
]['datetime_utc'].min()

co2_max_date = df_co2[
    (df_co2['user_id'] == 'id019') & 
    (df_co2['device'] == 'air control')
]['datetime_utc'].max()

df_co2 = df_co2[(df_co2['datetime_utc'] > co2_min_date) & (df_co2['datetime_utc'] < co2_max_date)]

co2_avg_sensor = float(df_co2[(df_co2['user_id']=='id019') & (df_co2['device']=='room sensor')]['co2 (ppm)'].mean())
co2_avg_aircontrol = float(df_co2[(df_co2['user_id']=='id019') & (df_co2['device']=='air control')]['co2 (ppm)'].mean())

offsets['co2'][user] = co2_avg_sensor - co2_avg_aircontrol

fig = px.scatter(
    df_co2[df_co2['user_id']== 'id019'], 
    x='datetime_utc', 
    y='co2 (ppm)', 
    color='device', 
    opacity=0.2, 
    title= f'Comparison of room sensor and aircontrol co2 measurements for user id019'
)

fig.add_hline(
    y=co2_avg_sensor,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg sensor {co2_avg_sensor:.2f}',
    annotation_position='top left'
)

fig.add_hline(
    y=co2_avg_aircontrol,
    line_dash='dash',
    line_color='blue',
    annotation_text=f'Avg aircontrol {co2_avg_aircontrol:.2f}',
    annotation_position='top left'
)

fig.write_html(f'id019_co2.html')
print(f'Sensor co2 was on avg {sum(list(offsets['co2'].values())) / len(list(offsets['co2'].values())):.2f} ppm greater than aircontrol co2')